<a href="https://colab.research.google.com/github/IKKNIGHT/Protein_interactions/blob/main/Phase3_4_Hotspots_and_Contigs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phases 3 + 4: hotspot arms, target cropping and RFdiffusion contigs
Produces, for every benchmark target, four **design specs** that differ **only in which hotspots are used**:

| Arm | Hotspots come from |
|---|---|
| `gnn` | the v3 GNN residue scores (`results/gnn_scores_v3/`) |
| `oracle` | the native interface (largest buried surface area), the upper bound |
| `baseline` | SASA x hydrophobicity (the simple baseline) |
| `scrambled` | a random compact site >= 15 A from the native interface (the negative control) |

Everything else (site size, cropping rule, binder length range, RFdiffusion settings) is identical across arms.

**Design change from the earlier plan:** hotspots are chosen as **one compact site** (best-scoring residues within 10 A of a seed), not spread over the surface. RFdiffusion binders target a single patch, and hotspots on opposite faces cannot all be satisfied.

Outputs go to `Protein_interactions/design_specs/`. No GPU needed.

In [1]:
# ── Cell 1: Mount, paths ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, re, subprocess, random
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'biopython', 'scipy', 'pandas'])
BASE = '/content/drive/MyDrive/Protein_interactions'
CONFIG, GOLD, RESULTS = (f'{BASE}/{d}' for d in ('config', 'gold', 'results'))
SPEC_DIR = f'{BASE}/design_specs'; os.makedirs(SPEC_DIR, exist_ok=True)
assert os.path.isdir(f'{RESULTS}/gnn_scores_v3'), 'run the Phase 2/2b notebook (Cell 10) first'

# ---- settings (identical for every arm) ----
SEED           = 42
N_HOTSPOTS     = 5       # RFdiffusion works best with ~3-6 hotspots in one patch
SITE_RADIUS    = 10.0    # A: all hotspots within this of the site seed
MIN_SEP        = 4.0     # A: avoid redundant neighbouring hotspots
# RATIONALE (SITE_RADIUS, MIN_SEP): ~10 A roughly matches a typical single-
# epitope patch size; 4 A separation is just over one Ca-Ca spacing (~3.8 A)
# so adjacent residues are not picked as two "distinct" hotspots. Neither
# value has a project-specific derivation or sensitivity sweep recorded in
# this repo -- thresholds chosen empirically; no derivation recorded.
HOTSPOT_MIN_SASA = 20.0  # A^2: a hotspot must be clearly exposed
# RATIONALE: 20 A^2 absolute SASA is a commonly used "meaningfully exposed
# residue" convention in computational PPI pipelines; no project-specific
# derivation or sensitivity sweep is recorded in this repo.
MAX_TARGET_RES = 180     # cropped target size cap
NUM_DESIGNS    = 50
RFD_WEIGHTS    = '/content/RFdiffusion/models/Complex_base_ckpt.pt'
RFD_SCRIPT     = '/content/RFdiffusion/scripts/run_inference.py'
RFD_OUT        = '/content/rfdiff_outputs'
random.seed(SEED)

Mounted at /content/drive


In [2]:
import numpy as np
from scipy.spatial import cKDTree

SMALL3 = {'GLY', 'ALA', 'SER', 'THR', 'PRO'}


def rank01(x, mask):
    # average ranks for ties (a plain argsort ramp gave tied zeros spuriously high ranks, which broke the oracle arm)
    from scipy.stats import rankdata
    out = np.zeros(len(x)); idx = np.where(mask)[0]
    if len(idx):
        out[idx] = rankdata(np.asarray(x)[idx], method='average') / len(idx)
    return out


def pick_site(score, cen, exposed, resnames, k=5, radius=10.0, min_sep=4.0, seed=None):
    # ONE compact binding site: seed = residue whose exposed neighbourhood has the best mean (rank-normalised)
    # score; hotspots = the k best-scoring exposed residues within `radius` of the seed, >= min_sep apart.
    # (RFdiffusion hotspots should sit in one patch, so they are deliberately NOT spread over the surface.)
    w = np.array([0.6 if r in SMALL3 else 1.0 for r in resnames])
    s = rank01(score, exposed) * w
    idx = np.where(exposed)[0]
    if len(idx) == 0: return []
    if seed is None:
        nbrs = cKDTree(cen[idx]).query_ball_point(cen[idx], radius)
        seed = idx[int(np.argmax([s[idx][j].mean() for j in nbrs]))]
    cand = idx[np.linalg.norm(cen[idx] - cen[seed], axis=1) <= radius]
    cand = cand[np.argsort(-s[cand])]
    chosen = []
    for c in cand:
        if all(np.linalg.norm(cen[c] - cen[o]) >= min_sep for o in chosen): chosen.append(int(c))
        if len(chosen) == k: break
    return chosen


def crop_target(chain, resnum, icode, cen, hot, max_res=180, radii=(18, 16, 14, 12, 10, 8), fill=4, min_seg=4):
    # Keep residues within r of any hotspot (shrinking r until <= max_res), bridge short gaps, then split into
    # contiguous segments (numbering must step by exactly 1 within a chain). Insertion-code residues are dropped
    # because a contig string cannot express them.
    n = len(chain); hot = list(hot)
    ok = np.array([ic == '' for ic in icode])
    d = np.min(np.linalg.norm(cen[:, None, :] - cen[hot][None, :, :], axis=2), axis=1)
    for r in radii:
        keep = (d <= r) & ok; keep[hot] = True
        i = 0
        while i < n:                                   # bridge short interior gaps of consecutive residues
            if not keep[i]:
                j = i
                while j < n and not keep[j]: j += 1
                if 0 < i and j < n and j - i <= fill and chain[i - 1] == chain[j] and resnum[j] - resnum[i - 1] == j - i + 1 \
                        and all(chain[t] == chain[i - 1] for t in range(i, j)):
                    keep[i:j] = ok[i:j]
                i = j
            else: i += 1
        if keep.sum() <= max_res: break
    segs, cur = [], None
    for i in range(n):
        if not keep[i]: cur = None; continue
        if cur is not None and chain[i] == cur['chain'] and resnum[i] == cur['end'] + 1 and cur['last'] == i - 1:
            cur['end'] = resnum[i]; cur['last'] = i; cur['n'] += 1; cur['idx'].append(i)
        else:
            cur = {'chain': chain[i], 'start': resnum[i], 'end': resnum[i], 'last': i, 'n': 1, 'idx': [i]}; segs.append(cur)
    hs = set(hot)
    segs = [s for s in segs if s['n'] >= min_seg or hs & set(s['idx'])]
    return [(s['chain'], s['start'], s['end']) for s in segs], int(sum(s['n'] for s in segs)), r


def build_contig(segments, lo, hi):
    return '/0 '.join(f'{c}{a}-{b}' for c, a, b in segments) + f'/0 {lo}-{hi}'


def binder_range(partner_len, iface_span, lo_min=40, hi_max=120):
    L = partner_len if partner_len <= hi_max else min(iface_span + 10, hi_max)
    lo = max(lo_min, int(round(0.7 * L)))
    return lo, max(lo + 20, min(hi_max, int(round(1.3 * L))))


# ---- quick self-test of the helpers ----
import numpy as np
_rng = np.random.default_rng(1); _n = 60
_ch, _rn = ['A'] * 30 + ['B'] * 30, list(range(1, 31)) + list(range(1, 31))
_cen = np.cumsum(_rng.normal(0, 2.2, (_n, 3)), axis=0); _ex = _rng.random(_n) > 0.3
_h = pick_site(_rng.random(_n), _cen, _ex, ['LEU'] * _n, k=5)
_segs, _n_in, _ = crop_target(_ch, _rn, [''] * _n, _cen, _h, max_res=40)
assert _h and all(_ex[i] for i in _h) and _n_in <= 40 + 5 and build_contig(_segs, 40, 60).endswith('/0 40-60')
print('Helper self-tests passed.')

Helper self-tests passed.


In [3]:
# ── Cell 3: Load every benchmark target (residues, exposure, GNN scores, native interface) ──
from Bio import PDB
from Bio.PDB.SASA import ShrakeRupley
import pandas as pd
from scipy.spatial import cKDTree

KD = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4, 'THR': -0.7,
      'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5, 'GLN': -3.5, 'ASP': -3.5,
      'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
hyd01 = lambda r: (KD.get(r, 0.0) + 4.5) / 9.0

def load_target_res(path):
    m = PDB.PDBParser(QUIET=True).get_structure('t', path)[0]
    s0 = PDB.Structure.Structure('s'); s0.add(m.copy())                    # model 0 only
    ShrakeRupley(probe_radius=1.4, n_points=100).compute(s0, level='R')
    out = {k: [] for k in ('chain', 'resnum', 'icode', 'name', 'cen', 'sasa', 'key')}
    for r in s0[0].get_residues():
        if r.id[0] != ' ': continue
        xyz = np.array([a.coord for a in r if (a.element or '').strip() != 'H'])
        ch, ic = r.get_parent().id, r.id[2].strip()
        out['chain'].append(ch); out['resnum'].append(r.id[1]); out['icode'].append(ic); out['name'].append(r.get_resname())
        out['cen'].append(xyz.mean(0)); out['sasa'].append(r.sasa); out['key'].append(f'{ch}:{r.id[1]}{ic}')
    out['cen'] = np.array(out['cen']); out['sasa'] = np.array(out['sasa'])
    return out

assign = json.load(open(f'{CONFIG}/chain_assignments.json'))
bench = {k: v for k, v in assign.items() if v['status'] in ('AUTO', 'CONFIRMED')}
T = {}
for key, v in bench.items():
    gdir = f'{GOLD}/{key}_{v["pdb_id"]}'; sp = f'{RESULTS}/gnn_scores_v3/{key}_{v["pdb_id"]}.json'
    if not (os.path.exists(f'{gdir}/interface_residues.json') and os.path.exists(sp)): print('skip', key, v['pdb_id']); continue
    info, sc, d = json.load(open(f'{gdir}/interface_residues.json')), json.load(open(sp)), load_target_res(f'{gdir}/target.pdb')
    keys, nat = d['key'], set(info['interface_consensus'])
    T[key] = dict(pdb=v['pdb_id'], target=v['target'], partner=v['partner'], gdir=gdir, d=d, info=info,
                  gnn=np.array([sc['gnn_post'].get(k, 0.0) for k in keys]),
                  hyd=np.array([hyd01(r) for r in d['name']]),
                  native=np.array([k in nat for k in keys]),
                  dsasa=np.array([info['dsasa_per_residue'].get(k, 0.0) for k in keys]))
print(len(T), 'targets loaded;', 'residues per target:', {t['target']: len(t['d']['key']) for t in T.values()})

20 targets loaded; residues per target: {'CDK2': 543, 'MDM2': 85, 'BRCA1': 103, 'RAD51': 210, 'PCNA': 248, 'BCL2': 137, 'BCL2L1 (Bcl-xL)': 175, 'FAS': 115, 'XIAP (BIR3)': 91, 'HRAS': 164, 'GNAI1': 320, 'ARF1': 165, 'JUN': 58, 'MYC': 88, 'CREBBP (KIX)': 81, 'MEN1': 467, 'VHL': 150, 'KEAP1': 285, 'CTNNB1': 508, 'VCL': 255}


In [4]:
# ── Cell 4: Hotspots for the four arms + hotspot precision against the native interface ──
rng = np.random.default_rng(SEED)

def far_seed(t):
    cen, exposed, native = t['d']['cen'], t['exposed'], t['native']
    if native.sum() == 0: return None
    dmin = cKDTree(cen[native]).query(cen)[0]
    far = np.where(exposed & (dmin >= 15.0))[0]
    return int(rng.choice(far)) if len(far) else int(np.where(exposed)[0][np.argmax(dmin[exposed])])

rows = []
for key, t in T.items():
    d = t['d']; t['exposed'] = (d['sasa'] >= HOTSPOT_MIN_SASA) & np.array([ic == '' for ic in d['icode']])
    seed_far = far_seed(t)
    scores = {'gnn': t['gnn'], 'oracle': t['dsasa'] * t['native'], 'baseline': d['sasa'] * t['hyd'], 'scrambled': rng.random(len(d['key']))}
    t['hot'] = {}
    for arm, sc in scores.items():
        t['hot'][arm] = pick_site(sc, d['cen'], t['exposed'], d['name'], k=N_HOTSPOTS, radius=SITE_RADIUS, min_sep=MIN_SEP,
                                  seed=seed_far if arm == 'scrambled' else None)
        h = t['hot'][arm]
        rows.append(dict(id=int(key), target=t['target'], arm=arm, n_hot=len(h),
                         precision=float(t['native'][h].mean()) if h else np.nan,
                         random_expect=float(t['native'][t['exposed']].mean())))
hp = pd.DataFrame(rows)
print('HOTSPOT PRECISION = fraction of the chosen hotspots that lie in the NATIVE interface (mean over targets)')
print(hp.groupby('arm')[['precision', 'random_expect']].mean().round(3).to_string())
pv = hp.pivot(index='target', columns='arm', values='precision')
print('\ngnn >= baseline on', int((pv['gnn'] >= pv['baseline']).sum()), '/', len(pv), 'targets;',
      'gnn > scrambled on', int((pv['gnn'] > pv['scrambled']).sum()), '/', len(pv))
print('\nPer-target hotspot precision:'); print(pv[['gnn', 'oracle', 'baseline', 'scrambled']].round(2).to_string())

HOTSPOT PRECISION = fraction of the chosen hotspots that lie in the NATIVE interface (mean over targets)
           precision  random_expect
arm                                
baseline        0.33          0.218
gnn             0.68          0.218
oracle          0.97          0.218
scrambled       0.00          0.218

gnn >= baseline on 18 / 20 targets; gnn > scrambled on 15 / 20

Per-target hotspot precision:
arm              gnn  oracle  baseline  scrambled
target                                           
ARF1             1.0     1.0       0.0        0.0
BCL2             1.0     0.8       0.2        0.0
BCL2L1 (Bcl-xL)  1.0     1.0       1.0        0.0
BRCA1            1.0     1.0       0.4        0.0
CDK2             1.0     1.0       1.0        0.0
CREBBP (KIX)     0.0     1.0       0.0        0.0
CTNNB1           0.0     1.0       0.0        0.0
FAS              0.6     1.0       1.0        0.0
GNAI1            0.6     1.0       0.4        0.0
HRAS             0.6     1.0      

In [5]:
# ── Cell 4b: Audit - are the GNN's "wrong" hotspots actually OTHER real interfaces? ──
# The GNN scores 'interface-likeness', not 'interface with THIS partner'. If a protein has a second binding site
# (VHL-Elongin C, KIX second site, beta-catenin alpha-catenin site ...) the GNN can be right about the surface and
# still 'wrong' against our single native partner. Here we find residues in contact with the OTHER chains that were
# present in the crystal structure (anything not in the target/partner assignment).
import requests
os.makedirs('/content/ppi_local', exist_ok=True)
def other_chain_contacts(pdb, tchains, pchains):
    p = f'/content/ppi_local/{pdb}.cif'
    if not os.path.exists(p): open(p, 'wb').write(requests.get(f'https://files.rcsb.org/download/{pdb}.cif', timeout=120).content)
    m = PDB.MMCIFParser(QUIET=True).get_structure(pdb, p)[0]
    heavy = lambda c: [a for r in c if r.id[0] == ' ' for a in r if (a.element or '').strip() != 'H']
    oth = [a for c in m if c.id not in set(tchains) | set(pchains) for a in heavy(c)]
    if not oth: return set(), 0
    tree = cKDTree(np.array([a.coord for a in oth])); out = set()
    for ch in tchains:
        for r in m[ch]:
            if r.id[0] != ' ': continue
            xyz = [a.coord for a in r if (a.element or '').strip() != 'H']
            if xyz and any(len(x) for x in tree.query_ball_point(xyz, 5.0)): out.add(f'{ch}:{r.id[1]}{r.id[2].strip()}')
    return out, len(oth)

rows = []
for key, t in T.items():
    a = bench[key]
    try: alt, n_oth = other_chain_contacts(t['pdb'], a['target_chains'], a['partner_chains'])
    except Exception as e: alt, n_oth = set(), -1
    keys = t['d']['key']; t['alt'] = np.array([(k in alt) and not n for k, n in zip(keys, t['native'])])
    h = t['hot']['gnn']
    rows.append(dict(target=t['target'], other_atoms=n_oth, alt_residues=int(t['alt'].sum()),
                     gnn_in_native=float(t['native'][h].mean()), gnn_in_other_interface=float(t['alt'][h].mean()),
                     gnn_in_either=float((t['native'][h] | t['alt'][h]).mean())))
au = pd.DataFrame(rows).set_index('target')
print(au.round(2).to_string())
print('\nTargets where the GNN site is a different REAL interface (not native, but in contact with another chain):')
print(au[(au.gnn_in_native < 0.4) & (au.gnn_in_other_interface >= 0.4)].index.tolist() or 'none')
print('Mean GNN hotspot precision: native only', round(au.gnn_in_native.mean(), 3), '| native or other real interface', round(au.gnn_in_either.mean(), 3))

                 other_atoms  alt_residues  gnn_in_native  gnn_in_other_interface  gnn_in_either
target                                                                                          
CDK2                       0             0            1.0                     0.0            1.0
MDM2                       0             0            0.8                     0.0            0.8
BRCA1                      0             0            1.0                     0.0            1.0
RAD51                     31             0            1.0                     0.0            1.0
PCNA                    4116            35            1.0                     0.0            1.0
BCL2                    1356             9            1.0                     0.0            1.0
BCL2L1 (Bcl-xL)            0             0            1.0                     0.0            1.0
FAS                    11977            30            0.6                     0.4            1.0
XIAP (BIR3)                0  

In [6]:
# ── Cell 5: Crop the target, build contigs and RFdiffusion commands, write design specs ──
def partner_span(info):
    spans = {}
    for k in info['partner_interface_5A']:
        m = re.match(r'(.+):(-?\d+)', k)
        if m: spans.setdefault(m.group(1), []).append(int(m.group(2)))
    return max([max(v) - min(v) + 1 for v in spans.values()], default=0)

specs, rows = [], []
for key, t in T.items():
    d = t['d']; lo, hi = binder_range(t['info']['partner_len'], partner_span(t['info']))
    for arm, h in t['hot'].items():
        if not h: continue
        radii = (1e4,) if len(d['key']) <= MAX_TARGET_RES else (18, 16, 14, 12, 10, 8)      # never crop a target that already fits
        segs, n_crop, radius = crop_target(d['chain'], d['resnum'], d['icode'], d['cen'], h, max_res=MAX_TARGET_RES, radii=radii)
        contig = build_contig(segs, lo, hi)
        hs = [f"{d['chain'][i]}{d['resnum'][i]}" for i in h]
        name = f'{key}_{t["pdb"]}_{arm}'
        cmd = (f"python {RFD_SCRIPT} inference.input_pdb={t['gdir']}/target.pdb "
               f"inference.output_prefix={RFD_OUT}/{name}/design inference.num_designs={NUM_DESIGNS} "
               f"\"contigmap.contigs=['{contig}']\" ppi.hotspot_res=\"[{','.join(hs)}]\" "
               f"inference.ckpt_override_path={RFD_WEIGHTS} denoiser.noise_scale_ca=0 denoiser.noise_scale_frame=0")
        spec = dict(name=name, id=int(key), pdb_id=t['pdb'], target=t['target'], partner=t['partner'], arm=arm,
                    target_pdb=f"{t['gdir']}/target.pdb", hotspots=hs, segments=segs, n_target_res_after_crop=n_crop,
                    crop_radius_A=radius, binder_len_range=[lo, hi], contig=contig, native_partner_len=t['info']['partner_len'],
                    hotspot_precision=float(t['native'][h].mean()), num_designs=NUM_DESIGNS, command=cmd)
        json.dump(spec, open(f'{SPEC_DIR}/{name}.json', 'w'), indent=2); specs.append(spec)
        rows.append(dict(name=name, target=t['target'], arm=arm, hotspots=' '.join(hs), crop_res=n_crop, n_segments=len(segs),
                         binder=f'{lo}-{hi}', precision=spec['hotspot_precision']))
summ = pd.DataFrame(rows); summ.to_csv(f'{SPEC_DIR}/summary.csv', index=False)
open(f'{SPEC_DIR}/commands.txt', 'w').write('\n\n'.join(f"# {s['name']}\n{s['command']}" for s in specs))
print(f'{len(specs)} design specs written to {SPEC_DIR}')
print('cropped target size:', summ.crop_res.describe()[['min', '50%', 'max']].astype(int).to_dict(),
      '| targets with >1 segment:', int((summ.n_segments > 1).sum()), 'of', len(summ))
print('\nGATE 1 - MDM2 / p53 specs:')
print(summ[summ.target == 'MDM2'].to_string(index=False))
m = [s for s in specs if s['target'] == 'MDM2' and s['arm'] == 'gnn']
if m: print('\nExample command:\n' + m[0]['command'])

80 design specs written to /content/drive/MyDrive/Protein_interactions/design_specs
cropped target size: {'min': 27, '50%': 107, 'max': 179} | targets with >1 segment: 42 of 80

GATE 1 - MDM2 / p53 specs:
            name target       arm              hotspots  crop_res  n_segments binder  precision
      5_1YCR_gnn   MDM2       gnn  A100 A99 A96 A54 A97        85           1  40-60        0.8
   5_1YCR_oracle   MDM2    oracle   A96 A93 A94 A99 A73        85           1  40-60        1.0
 5_1YCR_baseline   MDM2  baseline A109 A27 A104 A26 A47        85           1  40-60        0.4
5_1YCR_scrambled   MDM2 scrambled   A46 A45 A44 A39 A56        85           1  40-60        0.0

Example command:
python /content/RFdiffusion/scripts/run_inference.py inference.input_pdb=/content/drive/MyDrive/Protein_interactions/gold/5_1YCR/target.pdb inference.output_prefix=/content/rfdiff_outputs/5_1YCR_gnn/design inference.num_designs=50 "contigmap.contigs=['A25-109/0 40-60']" ppi.hotspot_res="[A100,A99